In [3]:
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [4]:
blob_centers = np.array([[ 0.2,  2.3], [-1.5 ,  2.3], [-2.8,  1.8],
                         [-2.8,  2.8], [-2.8,  1.3]])
blob_std = np.array([0.4, 0.3, 0.1, 0.1, 0.1])

X, y = make_blobs(n_samples=2000, centers=blob_centers, cluster_std=blob_std,
                  random_state=7)

In [5]:
k = 5
kmeans = KMeans(n_clusters=k, random_state=42)
y_pred = kmeans.fit_predict(X)

In [6]:
y_pred

array([2, 2, 4, ..., 1, 4, 2], dtype=int32)

In [7]:
y_pred is kmeans.labels_

True

In [8]:
# take a look at the centroids found
kmeans.cluster_centers_

array([[-0.066884  ,  2.10378803],
       [-2.79290307,  2.79641063],
       [-2.80214068,  1.55162671],
       [-1.47468607,  2.28399066],
       [ 0.47042841,  2.41380533]])

In [9]:
# can easily assign new instances to the cluster whose centroid is closest....
X_new = np.array([[0, 2], [3, 2], [-3, 3], [-3, 2.5]])
kmeans.predict(X_new)

array([0, 4, 1, 1], dtype=int32)

kmeans does not behave very well when the blobs have different diameters because all it cares about when assigning an instance to a cluster is the distance to the centroid. Instead of assigning each instance to a single cluster, hard clustering, it can be useful to give each instance a score per cluster -> soft clustering. The score can be the distance between the instance and the centroid or a a similarity score (or affinity), such as Gaussian radial basis function. In KMeans, the transform() method measures the distance from each instance to every centroid.....

In [10]:
# transform method in kmeans, measures the distance from every instance to every centroid
print(f"X_new shape: {X_new.shape}")
kmeans.transform(X_new).round(2)

X_new shape: (4, 2)


array([[0.12, 2.9 , 2.84, 1.5 , 0.63],
       [3.07, 5.85, 5.82, 4.48, 2.56],
       [3.07, 0.29, 1.46, 1.69, 3.52],
       [2.96, 0.36, 0.97, 1.54, 3.47]])

In [13]:
kmeans.predict(X_new)

array([0, 4, 1, 1], dtype=int32)

instead of using hard clustering, we could use the distances above from each centroid.... This transformation to producing a k-dimensional dataset can be a very efficient nonlinear dimensionality reduction technique...

**K-Means algorithm:**
- start by placing centroids randomly. Then label the instances, update the centroids, then label again, update, and so on until the centroids stop moving. 
- algo is guaranteed to converge since the mean squared distance between the instances and their closest centroids can only go down at each step, and since it cannot be nbegative, it is guaranteed to converge
- although th ealgo is guaranteed to converge, it may not converge to the right solution. whether it does or not depends on the centroid initialization. 

we can mitigate this risk by improving centroid initialization

**Centroid initialization methods:**
- if you happen to know approximately where the centroids should be, you can set the init hyperparameter to a numpy array containing the list of centroids...

In [15]:
good_init = np.array([[-3,3],[-3,2],[-3,1],[-1,2],[0,2]])
kmeans = KMeans(n_clusters=5, init=good_init, random_state=42)
kmeans.fit(X)

KMeans(init=array([[-3,  3],
       [-3,  2],
       [-3,  1],
       [-1,  2],
       [ 0,  2]]),
       n_clusters=5, random_state=42)

another solution is to run the algorithm multiple times with different random initializations, and keep the best solution. The number of random initialization sis controlled by the n_init hyperparameter. by default its equal to 10 when using  init="random", which means the kmeans algo runs 10 times when u call fit and sklearn keeps the best solution.  To know what the best solution is, it uses a performance metric called inertia...(A model's inertia is the sum of all squared distances between each instance x and the closest centroid predicted by the model)

In [16]:
# inertia
kmeans.inertia_

211.5985372581684

In [17]:
# score method returns the negative inertia (negative because a predictor's score() method must always respect sklearn's greater is better rule)
kmeans.score(X)

-211.5985372581684

- by default, the sklearn kmeans algorithm uses 'kmeans++' as the default initializer. an improvement to initialization that strategically spreads out the initial cluster centers so as to be able to find the clusters better and faster...the n_init default is 1. (good thing, can play around with increasing n_init but keeping init to kmeans++)
- to accelerate can try setting algorithm to 'elkan' -> (sometimes doesn't work)
- to cluster huge datasets that do not fit into memory -> use MiniBatchKMeans. 

**Limitations:**
- suboptimal as we need to run the algorithm several times to avoid suboptimal solutions
- kmeans also doesn't behave well when the clusters have varying sizes, different densitires, or nonspherical shapes